In [ ]:

from langgraph.graph import StateGraph, START, END
from langchain_openai import AzureChatOpenAI
from typing import TypedDict
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:

AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT_EUS2')
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_APIKEY_EUS2')
AZURE_OPENAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
LLM_DEPLOYMENT_NAME = os.getenv('LLM_DEPLOYMENT_NAME', os.getenv('MODEL_NAME'))


model = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    azure_deployment=LLM_DEPLOYMENT_NAME,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0.2,
)

In [ ]:
class BatsmanInput(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int
    sr : int
    bpb:float
    boundary_percentage:float
    summary:str

In [ ]:
def calculate_sr(state:BatsmanInput)->BatsmanInput:
    sr= (state['runs']/state['balls'])*100
    return {'sr':sr}

def calculate_bpb(state:BatsmanInput)->BatsmanInput:
    bpb = state['runs']/state['balls']
    return {'bpb':bpb}

def calculate_boundary_percentage(state:BatsmanInput)->BatsmanInput:
    boundary_percentage = ((state['fours'] * 4) + (state['sixes'] * 6)) / state['balls'] * 100
    return {'boundary_percentage':boundary_percentage}

def summary(state:BatsmanInput)->str:
    summaary =  f"The batsman scored {state['runs']} runs in {state['balls']} balls with a strike rate of {state['sr']:.2f}, boundary percentage of {state['boundary_percentage']:.2f}% and {state['fours']} fours and {state['sixes']} sixes."
    return {'summary':summaary}

In [ ]:
graph  = StateGraph(BatsmanInput)

graph.add_node('calculate_sr',calculate_sr)
graph.add_node('calculate_bpb',calculate_bpb)
graph.add_node('calculate_boundary_percentage',calculate_boundary_percentage)
graph.add_node('summary',summary)


In [ ]:
#edges
graph.add_edge(START,'calculate_sr')
graph.add_edge(START,'calculate_bpb')   
graph.add_edge(START,'calculate_boundary_percentage')
graph.add_edge('calculate_sr','summary')
graph.add_edge('calculate_bpb','summary')
graph.add_edge('calculate_boundary_percentage','summary')
graph.add_edge('summary',END)


workflow = graph.compile()

In [ ]:
initial_state = BatsmanInput(runs=120, balls=80, fours=15, sixes=5, sr=0, bpb=0.0, boundary_percentage=0.0, summary="")
final_state = workflow.invoke(initial_state)